In [ ]:
!pip install torch torchvision
!pip install sckikit-learn
!pip install grad-cam

In [ ]:
#Import all of the necessary libraries for modeling and computing evaluation metrics
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pickle

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
import time
random.seed(42)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [ ]:
#Use these objects early before creating the model, so that you can easily change CSV path, image directory and other model specific comparisons
CSV_PATH = "/Users/harryshield/Documents/Data Science/DS4002/Project 3/cleaned_data.csv"
IMAGE_DIR = "/Users/harryshield/Documents/Data Science/DS4002/Project 3/ISIC 2020 Images"
OUTPUT_DIR = '/Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet'
NUM_FOLDS = 5
BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 0
RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_POS_WEIGHT = True
#Uses the same random seed whenever needs to be used
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
#This is defining the model so that PyTorch is able to pull the images from the image directory and match it with identifiers that are in the csv dataset
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, str(row["image_name"]) + ".jpg")
        label = float(row["target"])

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)
 #Turns all of the images to a specific size for consistency    
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.9,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
#Where the model is actually determined depending on the pre-trained model being used. In this case it is the ResNet18
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 1)
    return model


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.39):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_probs = []

    for images, labels in tqdm(loader, desc="Validation", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_preds = (all_probs >= threshold).astype(int)
#Runs the evaluation metrics on how well the model is training and validating results
    metrics = {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else np.nan,
        "confusion_matrix": confusion_matrix(all_labels, all_preds)
    }

    return metrics

In [ ]:
def main():
    df = pd.read_csv(CSV_PATH)

    # Basic checks
    required_cols = {"image_name", "target"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}")

    df = df.dropna(subset=["image_name", "target"]).copy()
    df["target"] = df["target"].astype(int)

    # Make sure files exist
    df["exists"] = df["image_name"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_DIR, str(x) + ".jpg")))
    missing = df.loc[~df["exists"], "image_name"].tolist()
    if missing:
        print(f"Warning: {len(missing)} images are missing. They will be dropped.")
        df = df[df["exists"]].copy()


    df = df.drop(columns="exists").reset_index(drop=True)

    X = df["image_name"].values
    y = df["target"].values

    #This splits the dataset into 5 identical folds that can be used in all of the models and then these folds can be compared in statistical tests
    if not os.path.exists("folds.pkl"):
        print("Creating new folds...")

        skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        folds = list(skf.split(X, y))

        with open("folds.pkl", "wb") as f:
            pickle.dump(folds, f)

    else:
        print("Loading existing folds...")
        with open("folds.pkl", "rb") as f:
            folds = pickle.load(f)

    fold_results = []
#Helps tell how much time is left within the model running
    start_time = time.time()

    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        fold_start = time.time()
        print(f"\n========== Fold {fold}/{NUM_FOLDS} ==========")
    

        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()
#Making sure there is a training and a validation dataset for each of the folds
        train_dataset = SkinLesionDataset(train_df, IMAGE_DIR, transform=train_transform)
        val_dataset = SkinLesionDataset(val_df, IMAGE_DIR, transform=val_transform)

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS
        )

        model = build_model().to(DEVICE)

        # Handle imbalance
        if USE_POS_WEIGHT:
            num_pos = (train_df["target"] == 1).sum()
            num_neg = (train_df["target"] == 0).sum()

            if num_pos > 0:
                pos_weight_value = num_neg / num_pos
                pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)
                criterion = nn.BCEWithLogitsLoss()
            else:
                criterion = nn.BCEWithLogitsLoss()
        else:
            criterion = nn.BCEWithLogitsLoss()

        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        best_model_wts = copy.deepcopy(model.state_dict())
        best_f1 = -1.0

        for epoch in tqdm(range(NUM_EPOCHS), desc=f"Fold {fold} - Epochs"):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            val_metrics = evaluate(model, val_loader, criterion, DEVICE)

            print(
                f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Recall: {val_metrics['recall']:.4f} | "
                f"Val F1: {val_metrics['f1']:.4f} | "
                f"Val AUC: {val_metrics['roc_auc']:.4f}"
            )

            if val_metrics["f1"] > best_f1:
                best_f1 = val_metrics["f1"]
                best_model_wts = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_model_wts)
        final_metrics = evaluate(model, val_loader, criterion, DEVICE)

        model.eval()

# pick 1 sample from validation set
        fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)
        
        NUM_SAMPLES_TO_SAVE = 5

# reproducible randomness per fold
        random.seed(42 + fold)

        indices = random.sample(
            range(len(val_dataset)),
            k=min(NUM_SAMPLES_TO_SAVE, len(val_dataset))
)
#Places GradCAM heatmaps over the image, to show where the model is placing specific importance
        for i in indices:
            sample_img, sample_label = val_dataset[i]
            input_tensor = sample_img.unsqueeze(0).to(DEVICE)

            rgb_img = sample_img.permute(1, 2, 0).cpu().numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            rgb_img = (rgb_img * std) + mean
            rgb_img = np.clip(rgb_img, 0, 1)

            logits = model(input_tensor)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= 0.39)

            target_layers = [model.layer4[-1]]
            targets = [ClassifierOutputTarget(0)]

            with GradCAM(model=model, target_layers=target_layers) as cam:
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

            visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

            filename = f"fold_{fold}_img_{i}_true_{int(sample_label.item())}_pred_{pred}_prob_{prob:.2f}.png"
            save_path = os.path.join(fold_dir, filename)

            print("Saving to:", save_path)

            plt.figure(figsize=(4, 4))
            plt.imshow(visualization)
            plt.title(f"Fold {fold} | T:{int(sample_label.item())} P:{pred} ({prob:.2f})")
            plt.axis("off")
            plt.savefig(save_path, bbox_inches="tight")
            plt.close()
#Showing all of the evaluation metrics
        print("\nBest fold metrics:")
        print(f"Accuracy:  {final_metrics['accuracy']:.4f}")
        print(f"Precision: {final_metrics['precision']:.4f}")
        print(f"Recall:    {final_metrics['recall']:.4f}")
        print(f"F1:        {final_metrics['f1']:.4f}")
        print(f"ROC-AUC:   {final_metrics['roc_auc']:.4f}")
        print("Confusion Matrix:")
        print(final_metrics["confusion_matrix"])

            # Fold timing
        fold_time = time.time() - fold_start
        print(f"\nFold {fold} time: {fold_time/60:.2f} minutes")

        # Total + ETA
        elapsed = time.time() - start_time
        avg_time_per_fold = elapsed / fold
        remaining_folds = NUM_FOLDS - fold
        eta = avg_time_per_fold * remaining_folds

        print(f"Elapsed time: {elapsed/60:.2f} minutes")
        print(f"Estimated time remaining: {eta/60:.2f} minutes")

        fold_results.append(final_metrics)

    # Average across folds
    avg_accuracy = np.mean([r["accuracy"] for r in fold_results])
    avg_precision = np.mean([r["precision"] for r in fold_results])
    avg_recall = np.mean([r["recall"] for r in fold_results])
    avg_f1 = np.mean([r["f1"] for r in fold_results])
    avg_auc = np.nanmean([r["roc_auc"] for r in fold_results])

    print("\n========== CROSS-VALIDATION SUMMARY ==========")
    print(f"Mean Accuracy:  {avg_accuracy:.4f}")
    print(f"Mean Precision: {avg_precision:.4f}")
    print(f"Mean Recall:    {avg_recall:.4f}")
    print(f"Mean F1:        {avg_f1:.4f}")
    print(f"Mean ROC-AUC:   {avg_auc:.4f}")

if __name__ == "__main__":
    main()



Creating new folds...

========== Fold 1/5 ==========


Fold 1 - Epochs:  20%|██        | 1/5 [04:13<16:55, 253.95s/it]

Epoch 1/5 | Train Loss: 0.5609 | Val Loss: 0.5692 | Val Acc: 0.6696 | Val Recall: 0.9478 | Val F1: 0.7415 | Val AUC: 0.8267


Fold 1 - Epochs:  40%|████      | 2/5 [06:52<09:54, 198.01s/it]

Epoch 2/5 | Train Loss: 0.3987 | Val Loss: 0.5585 | Val Acc: 0.7217 | Val Recall: 0.9304 | Val F1: 0.7698 | Val AUC: 0.8639


...

Fold 5 - Epochs:  80%|████████  | 4/5 [10:37<02:39, 159.22s/it]

Epoch 4/5 | Train Loss: 0.2648 | Val Loss: 0.5964 | Val Acc: 0.7696 | Val Recall: 0.9652 | Val F1: 0.8073 | Val AUC: 0.8641


Fold 5 - Epochs: 100%|██████████| 5/5 [13:17<00:00, 159.48s/it]


Epoch 5/5 | Train Loss: 0.2170 | Val Loss: 0.7577 | Val Acc: 0.7565 | Val Recall: 0.9739 | Val F1: 0.8000 | Val AUC: 0.8691


Saving to: /Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet/fold_5/fold_5_img_90_true_0_pred_0_prob_0.26.png
Saving to: /Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet/fold_5/fold_5_img_16_true_1_pred_1_prob_1.00.png
Saving to: /Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet/fold_5/fold_5_img_110_true_1_pred_1_prob_0.99.png
Saving to: /Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet/fold_5/fold_5_img_141_true_0_pred_0_prob_0.02.png
Saving to: /Users/harryshield/Documents/Data Science/DS4002/Project 3/Gradcam_Outputs_ResNet/fold_5/fold_5_img_116_true_1_pred_1_prob_0.42.png

Best fold metrics:
Accuracy:  0.7696
Precision: 0.6937
Recall:    0.9652
F1:        0.8073
ROC-AUC:   0.8641
Confusion Matrix:
[[ 66  49]
 [  4 111]]

Fold 5 time: 13.69 minutes
Elapsed time: 70.11 minutes
Estimated time remaining: 0.00 minutes

========== CROSS-VALIDATION SUMMARY ====